# Ejemplo guiado: análisis de mediciones de equipos

Este notebook desarrolla el ejemplo guiado sobre las temáticas del Grupo 2:

1. Tabla de mediciones de equipos y tabla de información de equipos.
2. Filtro de condiciones críticas.
3. Creación de una variable de estado.
4. Indicadores por equipo con `groupby`.
5. Tabla pivote.
6. Unión (`merge`) de las dos tablas.
7. Transformación de una parte de los datos de formato ancho a largo (`melt`).

El escenario: un área de mantenimiento le toma mediciones de **temperatura** y **vibración**
a 4 equipos industriales durante 3 meses, y queremos analizar cuáles equipos están
entrando en condición crítica.

## 0. Importar librerías

In [1]:
import pandas as pd
import numpy as np

## 1. Tablas base

- `df_mediciones`: cada fila es una medición (temperatura y vibración) tomada a un equipo en una fecha.
- `df_equipos`: información fija de cada equipo (nombre, área, fecha de instalación).

Las dos tablas se pueden relacionar más adelante por la columna `id_equipo`.

In [2]:
# Tabla 1: mediciones tomadas a los equipos
df_mediciones = pd.DataFrame({
    'id_medicion': range(1, 13),
    'id_equipo': ['E-01', 'E-01', 'E-01', 'E-02', 'E-02', 'E-02',
                  'E-03', 'E-03', 'E-03', 'E-04', 'E-04', 'E-04'],
    'fecha': pd.to_datetime(['2025-01-05', '2025-02-05', '2025-03-05'] * 4),
    'temperatura': [72, 78, 85, 65, 68, 70, 90, 95, 98, 60, 62, 64],       # en °C
    'vibracion':   [4.2, 5.1, 7.8, 3.0, 3.2, 3.5, 6.9, 7.5, 8.2, 2.1, 2.3, 2.5]  # en mm/s
})

df_mediciones

,id_medicion,id_equipo,fecha,temperatura,vibracion
0,1,E-01,2025-01-05,72,4.2
1,2,E-01,2025-02-05,78,5.1
2,3,E-01,2025-03-05,85,7.8
3,4,E-02,2025-01-05,65,3.0
4,5,E-02,2025-02-05,68,3.2
5,6,E-02,2025-03-05,70,3.5
6,7,E-03,2025-01-05,90,6.9
7,8,E-03,2025-02-05,95,7.5
8,9,E-03,2025-03-05,98,8.2
9,10,E-04,2025-01-05,60,2.1


In [3]:
# Tabla 2: información de los equipos
df_equipos = pd.DataFrame({
    'id_equipo': ['E-01', 'E-02', 'E-03', 'E-04'],
    'nombre': ['Bomba centrífuga', 'Motor eléctrico', 'Compresor de aire', 'Ventilador industrial'],
    'area': ['Planta 1', 'Planta 1', 'Planta 2', 'Planta 2'],
    'fecha_instalacion': pd.to_datetime(['2019-06-01', '2020-03-15', '2018-11-20', '2021-01-10'])
})

df_equipos

,id_equipo,nombre,area,fecha_instalacion
0,E-01,Bomba centrífuga,Planta 1,2019-06-01
1,E-02,Motor eléctrico,Planta 1,2020-03-15
2,E-03,Compresor de aire,Planta 2,2018-11-20
3,E-04,Ventilador industrial,Planta 2,2021-01-10


## 2. Filtrar condiciones críticas

Se definen umbrales de temperatura y vibración. Una medición se considera crítica si
supera **cualquiera** de los dos límites (por eso se usa el operador `|`, no hace falta
que fallen los dos al mismo tiempo).

In [4]:
# Umbrales de criticidad
LIMITE_TEMP = 80        # °C
LIMITE_VIBRACION = 7.1  # mm/s, tomado como referencia de severidad alta (ISO 10816)

condicion_critica = (df_mediciones['temperatura'] > LIMITE_TEMP) | (df_mediciones['vibracion'] > LIMITE_VIBRACION)

df_criticos = df_mediciones[condicion_critica]
df_criticos

,id_medicion,id_equipo,fecha,temperatura,vibracion
2,3,E-01,2025-03-05,85,7.8
6,7,E-03,2025-01-05,90,6.9
7,8,E-03,2025-02-05,95,7.5
8,9,E-03,2025-03-05,98,8.2


## 3. Variable de estado

Con la misma condición del paso anterior se crea una columna nueva en `df_mediciones`
para marcar cada fila como `'Normal'` o `'Crítico'`.

In [5]:
df_mediciones['estado'] = np.where(condicion_critica, 'Crítico', 'Normal')
df_mediciones

,id_medicion,id_equipo,fecha,temperatura,vibracion,estado
0,1,E-01,2025-01-05,72,4.2,Normal
1,2,E-01,2025-02-05,78,5.1,Normal
2,3,E-01,2025-03-05,85,7.8,Crítico
3,4,E-02,2025-01-05,65,3.0,Normal
4,5,E-02,2025-02-05,68,3.2,Normal
5,6,E-02,2025-03-05,70,3.5,Normal
6,7,E-03,2025-01-05,90,6.9,Crítico
7,8,E-03,2025-02-05,95,7.5,Crítico
8,9,E-03,2025-03-05,98,8.2,Crítico
9,10,E-04,2025-01-05,60,2.1,Normal


## 4. Indicadores por equipo con `groupby`

Se agrupan las mediciones por `id_equipo` para calcular, por cada equipo:

- el promedio de temperatura,
- el promedio de vibración,
- el total de mediciones tomadas,
- cuántas de esas mediciones fueron críticas.

In [6]:
indicadores_equipo = df_mediciones.groupby('id_equipo').agg(
    temp_promedio=('temperatura', 'mean'),
    vibracion_promedio=('vibracion', 'mean'),
    mediciones_totales=('id_medicion', 'count'),
    mediciones_criticas=('estado', lambda x: (x == 'Crítico').sum())
).reset_index()

indicadores_equipo

,id_equipo,temp_promedio,vibracion_promedio,mediciones_totales,mediciones_criticas
0,E-01,78.333333,5.700000,3,1
1,E-02,67.666667,3.233333,3,0
2,E-03,94.333333,7.533333,3,3
3,E-04,62.000000,2.300000,3,0


## 5. Tabla pivote

Se construye una tabla de doble entrada: un equipo por fila, una fecha por columna,
y la temperatura como valor. Sirve para ver de un vistazo cómo evoluciona la
temperatura de cada equipo mes a mes.

In [7]:
tabla_pivote = pd.pivot_table(
    df_mediciones,
    index='id_equipo',
    columns='fecha',
    values='temperatura',
    aggfunc='mean'
)

tabla_pivote

fecha,2025-01-05,2025-02-05,2025-03-05
id_equipo,,,
E-01,72.0,78.0,85.0
E-02,65.0,68.0,70.0
E-03,90.0,95.0,98.0
E-04,60.0,62.0,64.0


## 6. Unión de tablas con `merge`

Se combinan los indicadores por equipo (Paso 4) con la información de cada equipo,
usando `id_equipo` como clave. Como todos los equipos de `indicadores_equipo`
también están en `df_equipos`, un `how='left'` trae todo sin perder filas.

In [8]:
df_resumen = pd.merge(indicadores_equipo, df_equipos, on='id_equipo', how='left')
df_resumen

,id_equipo,temp_promedio,vibracion_promedio,mediciones_totales,mediciones_criticas,nombre,area,fecha_instalacion
0,E-01,78.333333,5.700000,3,1,Bomba centrífuga,Planta 1,2019-06-01
1,E-02,67.666667,3.233333,3,0,Motor eléctrico,Planta 1,2020-03-15
2,E-03,94.333333,7.533333,3,3,Compresor de aire,Planta 2,2018-11-20
3,E-04,62.000000,2.300000,3,0,Ventilador industrial,Planta 2,2021-01-10


## 7. De formato ancho a largo con `melt()`

La `tabla_pivote` del Paso 5 está en formato ancho (una columna por fecha). Se vuelve
a formato largo, con una fila por cada combinación de equipo y fecha, que es la forma
que normalmente se necesita para graficar o para volver a analizar los datos.

In [9]:
tabla_pivote_reset = tabla_pivote.reset_index()

tabla_larga = tabla_pivote_reset.melt(
    id_vars='id_equipo',
    var_name='fecha',
    value_name='temperatura'
)

tabla_larga

,id_equipo,fecha,temperatura
0,E-01,2025-01-05 00:00:00,72.0
1,E-02,2025-01-05 00:00:00,65.0
2,E-03,2025-01-05 00:00:00,90.0
3,E-04,2025-01-05 00:00:00,60.0
4,E-01,2025-02-05 00:00:00,78.0
5,E-02,2025-02-05 00:00:00,68.0
6,E-03,2025-02-05 00:00:00,95.0
7,E-04,2025-02-05 00:00:00,62.0
8,E-01,2025-03-05 00:00:00,85.0
9,E-02,2025-03-05 00:00:00,70.0
